# Airplane Seat Capacity Analysis (1990-2025)

This notebook analyzes how airplane seat configurations have changed over 35 years using federal aviation data.

**Data Source:**
- Dataset: T-100 Domestic Segment (All Carriers)
- Database: Air Carrier Statistics (Form 41 Traffic)
- Office: Bureau of Transportation Statistics
- Department: United States Department of Transportation

**Key Metric:**  
Average seats per departure = Total seats / Total departures performed

This metric captures how airlines configure their aircraft - a rising average indicates airlines are fitting more seats into the same planes over time.

---
## 1. Load and Prepare Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
# The T-100 segment data contains every domestic flight segment reported to the DOT.
# Each row represents flights on a specific route, by a specific airline, using a specific aircraft type.
# The file is large (~800MB) because it spans 1990-2026 and includes all U.S. carriers.

df = pd.read_csv('../data/t100_segment_1990_2026.csv', low_memory=False)
df.shape

(13938270, 9)

In [3]:
# Check the year range and column names to confirm the data loaded correctly.

df['YEAR'].min(), df['YEAR'].max(), list(df.columns)

(np.int64(1990),
 np.int64(2026),
 ['DEPARTURES_PERFORMED',
  'SEATS',
  'PASSENGERS',
  'UNIQUE_CARRIER_NAME',
  'CARRIER_GROUP_NEW',
  'AIRCRAFT_TYPE',
  'AIRCRAFT_CONFIG',
  'YEAR',
  'MONTH'])

In [4]:
# The AIRCRAFT_TYPE column contains numeric codes, not names.
# This lookup table from BTS maps those codes to manufacturer and model names.

aircraft = pd.read_csv('../data/T_AIRCRAFT_TYPES.csv')
aircraft.head()

,AC_TYPEID,AC_GROUP,SSD_NAME,MANUFACTURER,LONG_NAME,SHORT_NAME,BEGIN_DATE,END_DATE
0,7,0,AERO COMMANDER 200,ROCKWELL,AERO COMMANDER 200,COMMANDR,1/1/1990 12:00:00 AM,NaN
1,8,0,AERO MACCHI AL-60,AERO MACCHI,AERO MACCHI AL-60,AL-60,1/1/1990 12:00:00 AM,NaN
2,9,0,AERONCA 7-AC,AERONCA,AERONCA 7-AC,AERONCA7,1/1/1990 12:00:00 AM,NaN
3,10,0,BEECH 35/36,BEECHCRAFT,BEECH BONANZA 35A/C/D/E/G/H/J/K/S/V/ 36A,BONANZA,1/1/1990 12:00:00 AM,NaN
4,20,0,BELLANCA CH-300,BELLANCA,BELLANCA CH-300,CH-300,1/1/1990 12:00:00 AM,NaN


In [5]:
# Join the flight data with the aircraft lookup table.
# AC_TYPEID in the lookup matches AIRCRAFT_TYPE in the flight data.
# After this join, each row will have the manufacturer name and aircraft model.

df = df.merge(
    aircraft[['AC_TYPEID', 'MANUFACTURER', 'LONG_NAME', 'SHORT_NAME']],
    left_on='AIRCRAFT_TYPE',
    right_on='AC_TYPEID',
    how='left'
)

df.shape

(13938270, 13)

In [6]:
# Filter to rows where flights actually occurred.
# Some rows have zero departures (scheduled but not operated routes).
# Also exclude rows where we couldn't match the aircraft type.

df = df[(df['DEPARTURES_PERFORMED'] > 0) & (df['MANUFACTURER'].notna())].copy()
df.shape

(13880358, 13)

---
## 2. Explore the Data

Before building charts, I need to understand what's in the data - which manufacturers and aircraft models appear most frequently.

In [7]:
# Which manufacturers have the most flight records?
# Boeing dominates, followed by regional jet makers (Embraer, Bombardier) and Airbus.

df['MANUFACTURER'].value_counts().head(15)

MANUFACTURER
BOEING                            5545930
AIRBUS INDUSTRIE                  2037498
MCDONNELL DOUGLAS                 1387808
EMBRAER                           1084141
CESSNA                            1043367
BOMBARDIER                         788334
CANADAIR                           367878
PIPER                              279829
BEECHCRAFT                         239907
DEHAVILLAND                        142967
SAAB-FAIRCHILD                     108160
FOKKER                              90325
DEHAVILLAND OF CANADA               90157
AEROSPATIALE/AERITALIA              83150
CONSTRUCCIONES AERONAUTICAS,SA      73621
Name: count, dtype: int64

In [8]:
# Which specific aircraft models appear most often?
# The Boeing 737 variants dominate the list - it's the workhorse of U.S. domestic aviation.

df['LONG_NAME'].value_counts().head(20)

LONG_NAME
BOEING 737-800                                  823966
AIRBUS INDUSTRIE A320-100/200                   761087
BOEING 757-200                                  629619
BOEING 737-700/700LR/MAX 7                      546127
BOEING 737-300                                  534438
AIRBUS INDUSTRIE A319                           520343
MCDONNELL DOUGLAS DC9 SUPER 80/MD81/82/83/88    503214
CANADAIR RJ-200ER /RJ-440                       478227
BOEING 727-200/231A                             457229
CESSNA 208 CARAVAN                              447652
CESSNA C206/207/209/210 STATIONAIR              421093
EMBRAER-145                                     375980
BOEING 767-300/300ER                            362165
EMBRAER ERJ-175                                 314098
CANADAIR RJ-700                                 312079
BOEING 737-100/200                              287580
MCDONNELL DOUGLAS DC-9-30                       233232
CANADAIR CRJ 900                                232326


---
## 3. Chart 1 Analysis: Boeing Aircraft Family Comparison (Slope Chart)

This analysis compares how seat configurations changed for different Boeing aircraft families between 1990 and 2025.

**Hypothesis:** The 737, which remains in active production, will show increasing seat density as airlines squeeze in more seats. Older families like the 747 and 767 may show declining averages as they're retired from passenger service or converted to cargo.

In [9]:
# Filter to Boeing aircraft only

boeing = df[df['MANUFACTURER'] == 'BOEING'].copy()
boeing.shape

(5545930, 13)

In [10]:
# Group individual aircraft models into families.
# For example, 737-300, 737-800, and 737 MAX all become "Boeing 737"

def get_boeing_family(name):
    """Categorize a Boeing aircraft model into its family."""
    if pd.isna(name):
        return None
    name = name.upper()
    if '737' in name:
        return 'Boeing 737'
    elif '757' in name:
        return 'Boeing 757'
    elif '767' in name:
        return 'Boeing 767'
    elif '777' in name:
        return 'Boeing 777'
    elif '747' in name:
        return 'Boeing 747'
    elif '727' in name:
        return 'Boeing 727'
    elif '717' in name:
        return 'Boeing 717'
    return None

boeing['FAMILY'] = boeing['LONG_NAME'].apply(get_boeing_family)
boeing = boeing[boeing['FAMILY'].notna()]

boeing['FAMILY'].value_counts()

FAMILY
Boeing 737    3074283
Boeing 757     668505
Boeing 727     526632
Boeing 767     512845
Boeing 747     417862
Boeing 777     182579
Boeing 717      79380
Name: count, dtype: int64

In [11]:
# Calculate average seats per departure for each Boeing family, by year.
# I'm summing total seats and total departures first, then dividing.
# This is more accurate than averaging the averages.

by_family = boeing.groupby(['FAMILY', 'YEAR']).agg(
    total_seats=('SEATS', 'sum'),
    total_departures=('DEPARTURES_PERFORMED', 'sum')
).reset_index()

by_family['avg_seats'] = (by_family['total_seats'] / by_family['total_departures']).round(1)
by_family.head(10)

,FAMILY,YEAR,total_seats,total_departures,avg_seats
0,Boeing 717,2000,3959842.0,35306.0,112.2
1,Boeing 717,2001,11524797.0,105679.0,109.1
2,Boeing 717,2002,15711501.0,135841.0,115.7
3,Boeing 717,2003,20888529.0,178342.0,117.1
4,Boeing 717,2004,25347898.0,222393.0,114.0
5,Boeing 717,2005,27995631.0,245424.0,114.1
6,Boeing 717,2006,30706478.0,270983.0,113.3
7,Boeing 717,2007,30402182.0,268210.0,113.4
8,Boeing 717,2008,29847643.0,260093.0,114.8
9,Boeing 717,2009,28768269.0,245088.0,117.4


In [12]:
# For the slope chart, I need 1990 and 2025 values only.
# I'm also filtering to families with significant departures (>1000) to avoid noise.

by_family_filtered = by_family[by_family['total_departures'] > 1000]
slope_data = by_family_filtered[by_family_filtered['YEAR'].isin([1990, 2025])][['FAMILY', 'YEAR', 'avg_seats']]

slope_pivot = slope_data.pivot(index='FAMILY', columns='YEAR', values='avg_seats').reset_index()
slope_pivot.columns = ['Aircraft Family', '1990', '2025']

# Only keep aircraft that have data in BOTH 1990 and 2025
slope_clean = slope_pivot.dropna()
slope_clean = slope_clean.sort_values('2025', ascending=False)
slope_clean

,Aircraft Family,1990,2025
2,Boeing 737,122.3,164.4
4,Boeing 757,187.8,136.6
5,Boeing 767,211.9,58.1
3,Boeing 747,326.0,37.1


In [13]:
# Calculate the change for each family to understand the story

slope_clean['Change'] = slope_clean['2025'] - slope_clean['1990']
slope_clean['Pct_Change'] = ((slope_clean['Change'] / slope_clean['1990']) * 100).round(1)
slope_clean

,Aircraft Family,1990,2025,Change,Pct_Change
2,Boeing 737,122.3,164.4,42.1,34.4
4,Boeing 757,187.8,136.6,-51.2,-27.3
5,Boeing 767,211.9,58.1,-153.8,-72.6
3,Boeing 747,326.0,37.1,-288.9,-88.6


### Chart 1 Findings

**Boeing 737:** Average seats increased from 122.3 (1990) to 164.4 (2025) — a gain of 42 seats (+34%). This reflects airlines packing more seats into each plane.

**Boeing 757, 767, 747:** All show declining averages. This doesn't mean seats were removed — it reflects these aircraft being retired from passenger service or converted to cargo operations. The remaining flights are a smaller, non-representative sample.

In [14]:
# Export for Datawrapper slope chart
# The format needs: Aircraft Family, 1990, 2025

slope_export = slope_clean[['Aircraft Family', '1990', '2025']]
slope_export.to_csv('../data/chart1_boeing_slope.csv', index=False)
slope_export

,Aircraft Family,1990,2025
2,Boeing 737,122.3,164.4
4,Boeing 757,187.8,136.6
5,Boeing 767,211.9,58.1
3,Boeing 747,326.0,37.1


---
## 4. Chart 2 Analysis: All Manufacturers Over Time (Small Multiples)

This analysis shows average seat capacity trends for all major aircraft manufacturers from 1990 to 2025.

In [15]:
# Calculate average seats per departure by manufacturer and year

by_mfr = df.groupby(['MANUFACTURER', 'YEAR']).agg(
    total_seats=('SEATS', 'sum'),
    total_departures=('DEPARTURES_PERFORMED', 'sum')
).reset_index()

by_mfr['avg_seats'] = (by_mfr['total_seats'] / by_mfr['total_departures']).round(1)
by_mfr.head()

,MANUFACTURER,YEAR,total_seats,total_departures,avg_seats
0,AERONCA,2002,0.0,16.0,0.0
1,AEROSPATIALE,2010,7486.0,62.0,120.7
2,AEROSPATIALE/AERITALIA,1990,588846.0,12320.0,47.8
3,AEROSPATIALE/AERITALIA,1991,2334029.0,47640.0,49.0
4,AEROSPATIALE/AERITALIA,1992,6355979.0,126804.0,50.1


In [16]:
# Identify manufacturers with consistent presence in the data.
# I'm keeping only those with at least 20 years of data and 100,000+ departures.
# This filters out small operators and one-off aircraft types.

mfr_coverage = by_mfr.groupby('MANUFACTURER').agg(
    years_present=('YEAR', 'nunique'),
    total_deps=('total_departures', 'sum')
).reset_index()

major_mfrs = mfr_coverage[
    (mfr_coverage['years_present'] >= 20) & 
    (mfr_coverage['total_deps'] > 100000)
]['MANUFACTURER'].tolist()

len(major_mfrs), major_mfrs

(21,
 ['AEROSPATIALE/AERITALIA',
  'AIRBUS INDUSTRIE',
  'BEECHCRAFT',
  'BOEING',
  'BOMBARDIER',
  'BRITISH AEROSPACE',
  'CANADAIR',
  'CESSNA',
  'CONSTRUCCIONES AERONAUTICAS,SA',
  'DEHAVILLAND',
  'DEHAVILLAND OF CANADA',
  'EMBRAER',
  'FAIRCHILD DORNIER',
  'FAIRCHILD SWEARINGEN',
  'LOCKHEED',
  'MCDONNELL DOUGLAS',
  'PILATUS',
  'PILATUS BRITTEN-NORMAN',
  'PIPER',
  'SAAB-FAIRCHILD',
  'SHORT BROS.'])

In [17]:
# Filter to major manufacturers and pivot for the chart

by_mfr_major = by_mfr[by_mfr['MANUFACTURER'].isin(major_mfrs)]
pivot = by_mfr_major.pivot(index='YEAR', columns='MANUFACTURER', values='avg_seats')
pivot = pivot.loc[1990:2025]
pivot.head()

MANUFACTURER,AEROSPATIALE/AERITALIA,AIRBUS INDUSTRIE,BEECHCRAFT,BOEING,BOMBARDIER,BRITISH AEROSPACE,CANADAIR,CESSNA,"CONSTRUCCIONES AERONAUTICAS,SA",DEHAVILLAND,...,EMBRAER,FAIRCHILD DORNIER,FAIRCHILD SWEARINGEN,LOCKHEED,MCDONNELL DOUGLAS,PILATUS,PILATUS BRITTEN-NORMAN,PIPER,SAAB-FAIRCHILD,SHORT BROS.
YEAR,,,,,,,,,,,,,,,,,,,,,
1990,47.8,239.5,11.9,147.1,NaN,63.6,0.0,0.0,19.0,39.2,...,25.5,NaN,17.9,279.7,134.2,NaN,NaN,NaN,0.0,35.7
1991,49.0,220.5,2.8,149.3,NaN,38.6,0.0,3.8,19.1,39.0,...,30.0,NaN,18.1,281.0,138.1,NaN,NaN,NaN,NaN,36.0
1992,50.1,214.0,18.5,150.5,NaN,39.6,0.0,2.6,19.1,39.8,...,29.8,NaN,18.0,282.3,141.1,NaN,NaN,NaN,33.9,36.0
1993,52.1,198.9,18.5,150.4,NaN,41.5,0.0,0.0,20.1,39.2,...,28.1,NaN,18.0,280.4,138.5,NaN,NaN,0.0,33.7,35.0
1994,54.6,201.8,18.4,149.8,NaN,47.4,54.6,0.0,NaN,37.8,...,27.8,30.0,18.1,278.1,132.6,NaN,NaN,0.0,33.7,33.2


In [18]:
# Export for Datawrapper small multiples

pivot.round(1).to_csv('../data/chart2_manufacturers_by_year.csv')
pivot.round(1).tail()

MANUFACTURER,AEROSPATIALE/AERITALIA,AIRBUS INDUSTRIE,BEECHCRAFT,BOEING,BOMBARDIER,BRITISH AEROSPACE,CANADAIR,CESSNA,"CONSTRUCCIONES AERONAUTICAS,SA",DEHAVILLAND,...,EMBRAER,FAIRCHILD DORNIER,FAIRCHILD SWEARINGEN,LOCKHEED,MCDONNELL DOUGLAS,PILATUS,PILATUS BRITTEN-NORMAN,PIPER,SAAB-FAIRCHILD,SHORT BROS.
YEAR,,,,,,,,,,,,,,,,,,,,,
2021,35.3,165.6,3.4,143.0,62.6,NaN,65.8,6.6,0.2,64.9,...,69.7,30.0,NaN,0.0,2.2,8.3,8.5,6.2,22.4,0.0
2022,33.7,169.7,3.4,150.8,63.3,NaN,65.9,6.6,0.3,64.0,...,70.5,30.8,NaN,0.0,0.7,8.3,8.4,6.3,27.1,0.0
2023,41.9,174.8,3.6,156.1,65.9,NaN,66.0,6.7,0.3,51.7,...,71.3,29.7,NaN,0.0,5.5,8.4,8.2,6.3,20.8,0.0
2024,45.2,178.3,4.0,156.9,66.2,NaN,65.9,7.0,0.4,60.1,...,70.9,29.8,NaN,0.0,17.2,8.1,9.3,6.6,15.1,0.0
2025,34.1,178.7,4.4,157.9,65.2,NaN,65.8,6.8,0.2,63.2,...,70.0,29.6,NaN,0.0,15.2,8.2,7.6,6.8,0.0,0.0


---
## 5. Key Numbers for the Article

These are the specific figures that will appear in the written piece. Each number traces directly to the analysis above.

In [19]:
# Boeing 737 change: 1990 to 2025

b737_1990 = slope_clean[slope_clean['Aircraft Family'] == 'Boeing 737']['1990'].values[0]
b737_2025 = slope_clean[slope_clean['Aircraft Family'] == 'Boeing 737']['2025'].values[0]
b737_change = b737_2025 - b737_1990
b737_pct = (b737_change / b737_1990) * 100

{
    'boeing_737_1990': round(b737_1990, 1),
    'boeing_737_2025': round(b737_2025, 1),
    'boeing_737_change_seats': round(b737_change, 1),
    'boeing_737_change_pct': round(b737_pct, 1)
}

{'boeing_737_1990': np.float64(122.3),
 'boeing_737_2025': np.float64(164.4),
 'boeing_737_change_seats': np.float64(42.1),
 'boeing_737_change_pct': np.float64(34.4)}

In [20]:
# Total records analyzed

{
    'total_flight_records': len(df),
    'years_covered': f"{df['YEAR'].min()} to {df['YEAR'].max()}",
    'manufacturers_in_chart2': len(major_mfrs)
}

{'total_flight_records': 13880358,
 'years_covered': '1990 to 2026',
 'manufacturers_in_chart2': 21}

---
## 6. Source Attribution

**For all charts:**

Source: Bureau of Transportation Statistics, U.S. Department of Transportation. T-100 Domestic Segment (All Carriers), Air Carrier Statistics (Form 41 Traffic), 1990-2025.